In [1]:
!pip install transformers -qq

In [2]:
!pip install sentencepiece==0.1.94

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.5/507.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sentencepiece: filename=sentencepiece-0.1.94-cp312-cp312-linux_x86_64.whl size=1431481 sha256=89ee342e912764302fc6d0da8d45a7e86b51ee3ef5cb7c99a6117f04c1ef1283
  Stored in directory: /root/.cache/pip/wheels/da/ba/bf/333b4e58c9b7f1bc20fb6f381a5be2944069722a95dbd88243
Successfully built sentencepiece
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Uninstalling sentencepiece-0.2.1:
      Successfully uninstalled sentencepiece-0.2.1


In [3]:
import torch
import json
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config

In [4]:
model = T5ForConditionalGeneration.from_pretrained('t5-large')
tokenizer = T5Tokenizer.from_pretrained('t5-large')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [5]:
device = torch.device('cpu')

In [6]:
display_architecture = True
if display_architecture == True:
  print(model.config)

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 4096,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": true,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "ea

In [7]:
if display_architecture == True:
  print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=4096, bias=False)
              (wo): Linear(in_features=4096, out_features=1024, bias=False)
              (d

In [8]:
if display_architecture == True:
  print(model.encoder)

T5Stack(
  (embed_tokens): Embedding(32128, 1024)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=1024, out_features=1024, bias=False)
            (k): Linear(in_features=1024, out_features=1024, bias=False)
            (v): Linear(in_features=1024, out_features=1024, bias=False)
            (o): Linear(in_features=1024, out_features=1024, bias=False)
            (relative_attention_bias): Embedding(32, 16)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseActDense(
            (wi): Linear(in_features=1024, out_features=4096, bias=False)
            (wo): Linear(in_features=4096, out_features=1024, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
            (act): ReLU()
          )
          (layer_norm): T5LayerNorm()
 

In [9]:
if display_architecture == True:
  print(model.decoder)

T5Stack(
  (embed_tokens): Embedding(32128, 1024)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=1024, out_features=1024, bias=False)
            (k): Linear(in_features=1024, out_features=1024, bias=False)
            (v): Linear(in_features=1024, out_features=1024, bias=False)
            (o): Linear(in_features=1024, out_features=1024, bias=False)
            (relative_attention_bias): Embedding(32, 16)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=1024, out_features=1024, bias=False)
            (k): Linear(in_features=1024, out_features=1024, bias=False)
            (v): Linear(in_features=1024, out_features=1024, bias=False)
            (o): Linear(in_features=1024, out_feat

In [10]:
if display_architecture == True:
  print(model.forward)

<bound method T5ForConditionalGeneration.forward of T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=4096, bias=False)
              (wo): Linear(in_features=4

In [11]:
import textwrap

In [12]:
def summarize(text, ml):
  preprocess_text = text.strip().replace("\n", "")
  t5_prepared_Text = "summarize: " + preprocess_text
  wrapped_t5_prepared_Text = textwrap.fill(t5_prepared_Text, width=70)
  print(wrapped_t5_prepared_Text)
  tokenized_text = tokenizer.encode(t5_prepared_Text, return_tensors="pt").to(device)
  summary_ids = model.generate(tokenized_text,
                               num_beams=4,
                               no_repeat_ngram_size=2,
                               min_length=30,
                               max_length=ml,
                               early_stopping=True)
  output = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
  return output

In [16]:
text="""
  We hold these truths to be self-evident, that all men are created equal,
  that they are endowed by their Creator with certain unalienable Rights,
  that among these are Life, Liberty, and the pursuit of Happiness.
  That to secure these rights, Governments are instituted among Men,
  deriving their just powers from the consent of the governed,
  That whenever any Form of Government becomes destructive of these ends,
  it is the Right of the People to alter or to abolish it, and to institute
  new Government, laying its foundation on such principles and organizing
  its powers in such form, as to them shall seem most likely to effect
  their Safety and Happiness.
  Prudence, indeed, will dictate that Governments
  long established should not be changed for light and transient causes;
  and accordingly all experience hath shown, that mankind are more disposed
  to suffer, while evils are sufferable, than to right themselves by abolishing
  the forms to which they are accustomed.
  But when a long train of abuses and
  usurpations, pursuing invariably the same Object evinces a design to reduce
  them under absolute Despotism, it is their right, it is their duty, to throw
  off such Government, and to provide new Guards for their future security.
  --Such has been the patient sufferance of these Colonies; and such is now
  the necessity which constrains them to alter their former Systems of
  Government.
  The history of the present King of Great Britain is a history of repeated
  injuries and usurpations, all having in direct object the establishment
  of an absolute Tyranny over these States.
  To prove this, let Facts
  be submitted to a candid world.
  """

In [ ]:
print("Number of characters:", len(text))
summary = summarize(text, 50)
wrapped_summary = textwrap.fill(summary, width=70)
print ("\nSummarized text: \n", wrapped_summary)

Number of characters: 1685
summarize: We hold these truths to be self-evident, that all men are
created equal,  that they are endowed by their Creator with certain
unalienable Rights,  that among these are Life, Liberty, and the
pursuit of Happiness.  That to secure these rights, Governments are
instituted among Men,  deriving their just powers from the consent of
the governed,  That whenever any Form of Government becomes
destructive of these ends,  it is the Right of the People to alter or
to abolish it, and to institute  new Government, laying its foundation
on such principles and organizing  its powers in such form, as to them
shall seem most likely to effect  their Safety and Happiness.
Prudence, indeed, will dictate that Governments  long established
should not be changed for light and transient causes;  and accordingly
all experience hath shown, that mankind are more disposed  to suffer,
while evils are sufferable, than to right themselves by abolishing
the forms to which they a

In [14]:
#Bill of Rights,V
text ="""
  No person shall be held to answer for a capital, or otherwise infamous crime,
  unless on a presentment or indictment of a Grand Jury, except in cases arising
  in the land or naval forces, or in the Militia, when in actual service
  in time of War or public danger; nor shall any person be subject for
  the same offense to be twice put in jeopardy of life or limb;
  nor shall be compelled in any criminal case to be a witness against himself,
  nor be deprived of life, liberty, or property, without due process of law;
  nor shall private property be taken for public use without just compensation.
  """

print("Number of characters:",len(text))
summary = summarize(text,50)
wrapped_summary = textwrap.fill(summary, width=70)
print ("\nSummarized text: \n", wrapped_summary)

Number of characters: 608
summarize: No person shall be held to answer for a capital, or
otherwise infamous crime,  unless on a presentment or indictment of a
Grand Jury, except in cases arising  in the land or naval forces, or
in the Militia, when in actual service  in time of War or public
danger; nor shall any person be subject for  the same offense to be
twice put in jeopardy of life or limb;  nor shall be compelled in any
criminal case to be a witness against himself,  nor be deprived of
life, liberty, or property, without due process of law;  nor shall
private property be taken for public use without just compensation.

Summarized text: 
 no person shall be held to answer for a capital, or otherwise infamous
crime, except in cases arising in the land or naval forces . nor shall
any person be subject for the same offense to be twice put in jeopar


In [15]:
#Montana Corporate Law
#https://corporations.uslegal.com/state-corporation-law/montana-corporation-law/#:~:text=Montana%20Corporation%20Law,carrying%20out%20its%20business%20activities.
text ="""The law regarding corporations prescribes that a corporation can
  be incorporated in the state of Montana to serve any lawful purpose. In the
  state of Montana, a corporation has all the powers of a natural person for
  carrying out its business activities. The corporation can sue and be sued in
  its corporate name. It has perpetual succession. The corporation can buy, sell
  or otherwise acquire an interest in a real or personal property. It can conduct
  business, carry on operations, and have offices and exercise the powers in a
  state, territory or district in possession of the US, or in a foreign country.
  It can appoint officers and agents of the corporation for various duties and
  fix their compensation.
  The name of a corporation must contain the word "corporation" or its
  abbreviation "corp." The name of a corporation should not be deceptively
  similar to the name of another corporation incorporated in the same state. It
  should not be deceptively identical to the fictitious name adopted by a foreign
  corporation having business transactions in the state.
  The corporation is formed by one or more natural persons by executing and
  filing articles of incorporation to the secretary of state of filing. The
  qualifications for directors are fixed either by articles of incorporation
  or bylaws. The names and addresses of the initial directors and purpose of
  incorporation should be set forth in the articles of incorporation. The
  articles of incorporation should contain the corporate name, the number of
  shares authorized to issue, a brief statement of the character of business
  carried out by the corporation, the names and addresses of the directors
  until successors are elected, and name and addresses of incorporators. The
  shareholders have the power to change the size of board of directors.
  """

print("Number of characters:",len(text))
summary = summarize(text,50)
wrapped_summary = textwrap.fill(summary, width=70)
print ("\nSummarized text: \n", wrapped_summary)

Number of characters: 1852
summarize: The law regarding corporations prescribes that a
corporation can  be incorporated in the state of Montana to serve any
lawful purpose. In the  state of Montana, a corporation has all the
powers of a natural person for  carrying out its business activities.
The corporation can sue and be sued in  its corporate name. It has
perpetual succession. The corporation can buy, sell  or otherwise
acquire an interest in a real or personal property. It can conduct
business, carry on operations, and have offices and exercise the
powers in a  state, territory or district in possession of the US, or
in a foreign country.  It can appoint officers and agents of the
corporation for various duties and  fix their compensation.  The name
of a corporation must contain the word "corporation" or its
abbreviation "corp." The name of a corporation should not be
deceptively  similar to the name of another corporation incorporated
in the same state. It  should not be deceptiv